# Fine-Tuning an LLM for Emotion Classification using LoRA and QLoRA

## Project Overview

This notebook fine-tunes a small open-source Large Language Model for a domain-specific task: **emotion classification from text**.

The model is fine-tuned using **QLoRA**, where the base model is loaded in 4-bit quantized form and trained using lightweight **LoRA adapters**.

## Dataset

Dataset used: `dair-ai/emotion`

The dataset contains English text samples labeled with six emotions:

- sadness
- joy
- love
- anger
- fear
- surprise

## Task

Given a user text, the fine-tuned model predicts one emotion label.

## Model

Base model: `TinyLlama/TinyLlama-1.1B-Chat-v1.0`

Fine-tuning method: QLoRA with LoRA adapters.

## Training Fix

The model is trained using answer-only loss.  
The prompt tokens are ignored during training loss calculation, so the model learns to generate the final emotion label only.

In [1]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes
!pip install -q pandas==2.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.7 MB/s eta 0:00:00


## 1. Import Required Libraries

This section imports all libraries needed for loading the dataset, preparing the model, applying LoRA adapters, training, and generating predictions.

In [2]:
import os
import torch
import pandas as pd

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    default_data_collator
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

## 2. Check GPU Availability

Since QLoRA fine-tuning requires GPU acceleration, this cell checks whether a GPU is available in Google Colab.

In [3]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. In Colab, go to Runtime > Change runtime type > T4 GPU.")

CUDA available: True
GPU: Tesla T4


## 3. Load the Public Dataset

The `dair-ai/emotion` dataset is loaded from Hugging Face Datasets.

For faster training in Colab, a smaller subset of the dataset is used.

In [4]:
dataset = load_dataset("dair-ai/emotion")

print(dataset)
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})
{'text': 'i didnt feel humiliated', 'label': 0}


## 4. Inspect Dataset Labels

The dataset labels are converted from numbers into emotion names.

In [5]:
label_names = dataset["train"].features["label"].names

print(label_names)

['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


## 5. Load Tokenizer

The tokenizer is loaded before formatting the dataset because the EOS token is needed while preparing the training text.

In [6]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Tokenizer loaded.
Pad token: </s>
EOS token: </s>


## 6. Convert Dataset into Instruction Format

The model is trained using an instruction-style prompt.

The prompt and answer are stored separately.  
During tokenization, the prompt part will be ignored in the loss calculation, and only the answer label will be learned.

In [7]:
def create_prompt(text):
    prompt = f"""### Instruction:
Classify the emotion of the following text into exactly one label from this list:
sadness, joy, love, anger, fear, surprise

Only output the emotion label.

### Text:
{text}

### Answer:
"""
    return prompt


def format_example(example):
    label_text = label_names[example["label"]]

    prompt = create_prompt(example["text"])
    full_text = prompt + label_text + tokenizer.eos_token

    return {
        "prompt": prompt,
        "answer": label_text,
        "text": full_text
    }


train_dataset = dataset["train"].shuffle(seed=42).select(range(700))
eval_dataset = dataset["validation"].shuffle(seed=42).select(range(100))

train_dataset = train_dataset.map(format_example)
eval_dataset = eval_dataset.map(format_example)

print(train_dataset[0]["text"])

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

### Instruction:
Classify the emotion of the following text into exactly one label from this list:
sadness, joy, love, anger, fear, surprise

Only output the emotion label.

### Text:
while cycling in the country

### Answer:
fear</s>


## 7. Load Base LLM with 4-bit Quantization

The base language model is loaded in 4-bit quantized form.

This is the QLoRA part of the project.

Instead of updating all model parameters, only LoRA adapter parameters will be trained.

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

print("Model loaded with 4-bit quantization.")

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded with 4-bit quantization.


## 8. Apply LoRA Configuration

LoRA adapters are added to the quantized model.

During training, only these adapter parameters are updated.

In [9]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## 9. Tokenize the Dataset with Answer-Only Labels

This is the main training fix.

The prompt tokens are masked with `-100`, so the model does not learn to repeat the prompt.

Only the answer label tokens are used for training loss.

In [10]:
max_length = 256

def tokenize_function(example):
    full_tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

    prompt_tokens = tokenizer(
        example["prompt"],
        truncation=True,
        max_length=max_length
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]

    labels = input_ids.copy()

    prompt_length = len(prompt_tokens["input_ids"])

    # Ignore prompt tokens during loss calculation
    labels[:prompt_length] = [-100] * prompt_length

    # Ignore padding tokens during loss calculation
    labels = [
        label if token != tokenizer.pad_token_id else -100
        for label, token in zip(labels, input_ids)
    ]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


tokenized_train = train_dataset.map(
    tokenize_function,
    remove_columns=train_dataset.column_names
)

tokenized_eval = eval_dataset.map(
    tokenize_function,
    remove_columns=eval_dataset.column_names
)

print(tokenized_train[0].keys())

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])


## 10. Check the Training Labels

This cell verifies that the model is only being trained on the answer label.

Most prompt tokens should show `-100`, while the answer label should remain visible.

In [11]:
sample = tokenized_train[0]

visible_label_ids = [
    token_id for token_id in sample["labels"]
    if token_id != -100
]

print("Decoded training target:")
print(tokenizer.decode(visible_label_ids))

Decoded training target:
fear


## 11. Set Up Training Arguments

The model is trained using QLoRA.

Only the LoRA adapter parameters are updated.

The number of steps is kept reasonable for Google Colab T4.

In [12]:
output_dir = "./emotion_qlora_lora_model"

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    max_steps=180,
    logging_steps=10,
    save_strategy="no",
    eval_strategy="no",
    fp16=True,
    report_to="none",
    optim="paged_adamw_8bit",
    warmup_ratio=0.03,
    max_grad_norm=0.3
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=default_data_collator
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## 12. Fine-Tune the Model

This cell starts the fine-tuning process.

In [13]:
trainer.train()

# Switch from training/checkpointing mode to clean inference mode.
model.eval()
model.config.use_cache = True
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Step,Training Loss
10,1.433273
20,1.053236
30,0.937463
40,0.801698
50,1.335478
60,0.619588
70,0.488317
80,0.385535
90,1.283018
100,1.179302


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

## 13. Save the Fine-Tuned LoRA Adapter

The trained LoRA adapter is saved locally in the Colab environment.

In [14]:
adapter_save_path = "./fine_tuned_emotion_lora_adapter"

model.save_pretrained(adapter_save_path)
tokenizer.save_pretrained(adapter_save_path)

print("Saved LoRA adapter to:", adapter_save_path)

Saved LoRA adapter to: ./fine_tuned_emotion_lora_adapter


## 14. Test the Fine-Tuned Model

The fine-tuned model is tested using new sample text inputs.

The model should generate one of the six emotion labels:

- sadness
- joy
- love
- anger
- fear
- surprise

In [15]:
def score_emotion_label(text, label):
    prompt = create_prompt(text)

    prompt_inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(model.device)
    label_ids = tokenizer(label + tokenizer.eos_token, return_tensors="pt", add_special_tokens=False)["input_ids"].to(model.device)

    input_ids = torch.cat([prompt_inputs["input_ids"], label_ids], dim=1)
    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits

    prompt_length = prompt_inputs["input_ids"].shape[-1]
    label_token_ids = input_ids[0, prompt_length:]
    label_logits = logits[0, prompt_length - 1:-1, :]
    label_log_probs = torch.log_softmax(label_logits, dim=-1)
    token_scores = label_log_probs.gather(1, label_token_ids.unsqueeze(1)).squeeze(1)

    return token_scores.mean().item()


def predict_emotion(text, return_scores=False):
    model.eval()

    scores = {
        label: score_emotion_label(text, label)
        for label in label_names
    }
    prediction = max(scores, key=scores.get)

    if return_scores:
        return prediction, scores

    return prediction

## 15. Generate Sample Outputs

This cell generates sample predictions from the fine-tuned model.

Take a screenshot of this output table and include it in the PDF document.

In [16]:
test_samples = [
    "I feel very sad and lonely today.",
    "I am so happy and excited about my success.",
    "I love spending time with my family.",
    "I am angry because nobody listened to me.",
    "I feel scared walking alone at night.",
    "I am surprised by this unexpected gift."
]

results = []

for text in test_samples:
    prediction = predict_emotion(text)
    results.append({
        "Input Text": text,
        "Predicted Emotion": prediction
    })

results_df = pd.DataFrame(results)

results_df

,Input Text,Predicted Emotion
0,I feel very sad and lonely today.,sadness
1,I am so happy and excited about my success.,joy
2,I love spending time with my family.,joy
3,I am angry because nobody listened to me.,anger
4,I feel scared walking alone at night.,fear
5,I am surprised by this unexpected gift.,surprise


## 16. Test with More Natural Sentences

This extra test uses more natural examples to show that the fine-tuned model can classify unseen inputs.

In [17]:
extra_test_samples = [
    "I finally got the job I wanted and I feel amazing!",
    "I miss my old friends and feel very lonely today.",
    "I am so angry that nobody listened to me.",
    "The dark alley made me feel unsafe and nervous.",
    "My parents surprised me with a birthday party.",
    "I care about her so much and enjoy every moment with her."
]

extra_results = []

for text in extra_test_samples:
    prediction = predict_emotion(text)
    extra_results.append({
        "Input Text": text,
        "Predicted Emotion": prediction
    })

extra_results_df = pd.DataFrame(extra_results)

extra_results_df

,Input Text,Predicted Emotion
0,I finally got the job I wanted and I feel amaz...,joy
1,I miss my old friends and feel very lonely today.,sadness
2,I am so angry that nobody listened to me.,anger
3,The dark alley made me feel unsafe and nervous.,fear
4,My parents surprised me with a birthday party.,surprise
5,I care about her so much and enjoy every momen...,sadness
